# GNN GF-Model Baseline (Multi-PNA-EU) — Colab 실행 노트북

**baseline과의 차이점**
- 엣지 피처: **log_amount, currency, payment_format** (base 3개) + **gf.parquet 22개 피처** = 25개 (ports 제외)
- 노드 피처: **account_node_features.csv** (train-only PageRank, in/out degree, in/out amount sum)
- 샘플러: **LinkNeighborLoader** (baseline과 동일)
- 데이터 로딩: `gnn/gf_model/baseline/data_loading.py` (formatted_transactions_gf.csv + gf.parquet join)

**실행 전 체크리스트**
- [ ] 런타임 → GPU (T4 이상) 설정
- [ ] Google Drive에 `Graph_AML/data/processed/gnn_inputs/` 아래 산출물 존재 확인 (formatted_transactions_gf.csv, account_node_features.csv, account_mapping.csv)
- [ ] Google Drive에 `Graph_AML/gnn/gf_model/gf.parquet` 존재 확인
- [ ] **2번 셀** 경로 설정 확인
- [ ] **5번 셀** 실험 설정 확인

## 0. GPU 확인

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("GPU가 없습니다. 런타임 유형 변경 → GPU로 전환하세요.")

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정 ← 여기만 수정

In [ ]:
from pathlib import Path

# ============================================================
# 팀원 Drive 구조에 맞게 수정하세요
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/Graph_AML")
REPO_URL   = "https://github.com/JKYUPSYCHE/Graph_AML"
BRANCH     = "gnn/gf_model"

EXPERIMENT = "GNN-GF-02"
RUN        = "r01"
# ============================================================

GNN_INPUTS = DRIVE_BASE / "data" / "processed" / "gnn_inputs"
GF_PARQUET = DRIVE_BASE / "gnn" / "gf_model" / "gf.parquet"
GNN_DIR    = DRIVE_BASE / "gnn"
RUN_DIR    = GNN_DIR / EXPERIMENT / RUN

for sub in ['logs', 'models', 'runs']:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)

required = [
    GNN_INPUTS / "formatted_transactions_gf.csv",
    GNN_INPUTS / "account_node_features.csv",
    GNN_INPUTS / "account_mapping.csv",
    GF_PARQUET,
]
all_ok = True
for f in required:
    exists = f.exists()
    print(f"{'[OK]' if exists else '[MISSING]'} {f.name}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("필수 파일이 없습니다. 체크리스트를 확인하세요.")

print("\n경로 설정 완료")
print("GNN_INPUTS :", GNN_INPUTS)
print("GF_PARQUET :", GF_PARQUET)
print("RUN_DIR    :", RUN_DIR)

## 3. 레포 클론 & PyG 설치

In [ ]:
import os

REPO_DIR = Path("/content/Graph_AML")

if not REPO_DIR.exists():
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

os.system(f"git -C {REPO_DIR} fetch origin")
os.system(f"git -C {REPO_DIR} checkout -B {BRANCH} origin/{BRANCH}")

branch = os.popen(f"git -C {REPO_DIR} branch --show-current").read().strip()
print("레포 준비 완료:", REPO_DIR)
print("현재 브랜치  :", branch)
assert branch == BRANCH, f"브랜치 체크아웃 실패: {branch}"

In [ ]:
import torch

torch_ver = torch.__version__.split('+')[0]
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f"설치 대상: torch={torch_ver}, cuda={cuda_tag}")

os.system("pip install -q torch_geometric")
os.system(f"pip install -q pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html")
os.system("pip install -q psutil tqdm tensorboard")
print("패키지 설치 완료")

## 4. 모듈 경로 설정

In [ ]:
import sys

BASELINE_GF_DIR = REPO_DIR / "gnn" / "gf_model" / "baseline"

os.chdir(BASELINE_GF_DIR)

for p in [str(REPO_DIR), str(BASELINE_GF_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("CWD        :", os.getcwd())
print("sys.path[0]:", sys.path[0])
print("sys.path[1]:", sys.path[1])

## 5. 실험 설정 ← 여기만 수정

In [ ]:
from types import SimpleNamespace

# ============================================================
# 실험 설정 — 필요에 따라 수정
# ============================================================
args = SimpleNamespace(
    # 모델
    model      = 'pna',
    data       = 'Small_HI',
    seed       = 42,

    # 학습
    n_epochs   = 100,
    patience   = 20,

    # LinkNeighborLoader 샘플링 파라미터
    batch_size = 4096,
    num_neighs = [100, 100],

    # Multi-PNA-EU 핵심 플래그
    emlps         = True,
    reverse_mp    = True,
    ports         = True,
    reverse_ports = True,
    tds           = False,
    ego           = False,

    # 저장/추론
    save_model  = True,
    unique_name = 'small_hi_gf_baseline_pna_lnl',
    finetune    = False,
    inference   = False,

    tqdm = False,
)
# ============================================================

data_config = {
    "paths": {
        "aml_data"      : str(DRIVE_BASE / "data"),
        "gnn_inputs"    : str(GNN_INPUTS),
        "gf_parquet"    : str(GF_PARQUET),
        "model_to_load" : str(RUN_DIR / "models"),
        "model_to_save" : str(RUN_DIR / "models"),
        "tb_log_dir"    : str(RUN_DIR / "runs"),
    }
}

print(f"모델      : {args.model}")
print(f"데이터    : {args.data}")
print(f"에폭      : {args.n_epochs}  |  patience: {args.patience}")
print(f"샘플러    : LinkNeighborLoader  batch_size={args.batch_size}, num_neighs={args.num_neighs}")
print(f"엣지 피처 : log_amount + currency + payment_format (base 3) + gf.parquet 22개 = 25개 (ports 제외)")
print(f"노드 피처 : train-only PageRank, in/out degree, in/out amount sum (5개)")
print(f"플래그    : emlps={args.emlps}, reverse_mp={args.reverse_mp}, ports={args.ports}, tds={args.tds}")
print(f"저장 경로 : {RUN_DIR}")

## 6. 데이터 로딩

In [ ]:
import time
import logging
from utils import set_seed
from util import logger_setup
from data_loading import get_data

LOG_DIR = RUN_DIR / 'logs'
logger_setup(log_dir=LOG_DIR, log_name=args.unique_name)
set_seed(args.seed)

logging.info("데이터 로딩 시작")
t1 = time.perf_counter()

tr_data, val_data, te_data, tr_inds, val_inds, te_inds = get_data(args, data_config)

t2 = time.perf_counter()
data_load_elapsed = t2 - t1
logging.info(f"데이터 로딩 완료: {data_load_elapsed:.2f}s")

## 7. 학습

> LinkNeighborLoader로 미니배치를 구성합니다.  
> `batch_size`와 `num_neighs`가 배치 크기와 이웃 샘플링 범위를 결정합니다.

In [ ]:
import psutil
import torch
from training import train_gnn

_m, _s = divmod(int(data_load_elapsed), 60)
print(f"데이터 로딩 소요 시간: {_m:02d}분 {_s:02d}초")

cpu_pct = psutil.cpu_percent(interval=1)
ram     = psutil.virtual_memory()
print(f"CPU 사용률 : {cpu_pct:.1f}%")
print(f"RAM        : {ram.used / 1024**3:.1f} / {ram.total / 1024**3:.1f} GB  ({ram.percent:.1f}%)")

if torch.cuda.is_available():
    dev   = torch.cuda.current_device()
    used  = torch.cuda.memory_allocated(dev) / 1024**3
    total = torch.cuda.get_device_properties(dev).total_memory / 1024**3
    print(f"GPU        : {torch.cuda.get_device_name(dev)}")
    print(f"GPU 메모리 : {used:.1f} / {total:.1f} GB")
else:
    print("GPU        : 없음 (CPU 모드)")

logging.info("학습 시작")
train_gnn(tr_data, val_data, te_data, tr_inds, val_inds, te_inds, args, data_config)

## 8. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN_DIR}/runs